# Clonal Selection in Normal Tissues

## Introduction
In this practical session, we will explore the signals of **positive selection** in somatic mutations of the normal bladder urothelium.

Clonal selection is the process by which certain mutations provide a fitness advantage to cells, allowing them to outcompete their neighbors and expand within a tissue. This is a fundamental process in both cancer development and the aging of normal tissues (clonal hematopoiesis, etc.).

We will proceed with the analysis in three main steps:

1.  **Mutation Landscape**: Analyzing the frequency and distribution of mutation types (synonymous, missense, truncating).

2.  **Signals of selection**: Using state-of-the-art tools to detect clonal selection:
    *   **$\omega$ (dN/dS)**: Do we observe more mutations than expected? (https://github.com/bbglab/omega)
    *   **OncodriveFML**: Are the mutations we are observing biased towards high functional impact? (https://github.com/bbglab/oncodrivefml)
    *   **Oncodrive3D**: Are the mutations detected clustered in the 3D space? (https://github.com/bbglab/oncodrive3d)

3.  **Epidemiological Correlation**: Investigating how these selection signals vary with patient metadata like **Age** and **Sex**.


## You will find some questions for discussion in the notebook, feel free to collect the answers either written in paper or in some slides where you can paste the plots.

### Fetch data

In [ ]:
!git clone https://github.com/WCSCourses/Cancer_Genomic_Epidemiology_2026.git

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns


# Set display options
pd.set_option('display.max_columns', None)

In [ ]:
data_path = "Cancer_Genomic_Epidemiology_2026/course_data_2026/Clonal_selection_data"
input_data_path = f"{data_path}/input"
output_data_path = f"{data_path}/output"
SAMPLE_NAME = "all_samples"

from Cancer_Genomic_Epidemiology_2026.course_modules_2026.Clonal_selection_demo.utils import generate_all_side_figures, get_all_data, get_counts_per_position_n_consequence, plot_count_track, plot_stacked_bar_track_binned, metrics_colors_dictionary


## 1. Select a subset of genes

In small groups, pick some genes from the available list. You may see more genes in the input files provided to the notebook, but these are the ones with complete information

In [ ]:
possible_genes = ['CREBBP', 'KMT2D', 'RBM10', 'TP53', 'PIK3CA', 'ARID1A',
                  'FGFR3', 
                  'EP300', 'KDM6A', 'FOXQ1',
                  'CDKN1A', 'NOTCH2', 'STAG2',
                  'RB1',
                  ]


### Download Data
*Instructions: Download the folder corresponding to your chosen gene. For this notebook, we assume the files are present in the current working directory.*

Define your chosen gene and sample name below. Based on the file structure, the "sample" name for the combined analysis is likely `all_samples`.

In [ ]:
# Define your gene and sample
GENE_LIST = ["KMT2D", "CREBBP", "RBM10"]  # Change this to your chosen gene (e.g., CREBBP, EP300, RBM10, etc.)


## 2. Analyzing the Mutation Landscape

Before jumping into complex selection metrics, it is crucial to understand the data behind all those computations.

We distinguish between:
*   **Synonymous**: Usually neutral.
*   **Missense**: Change one amino acid, not all amino acid changes at any position have the same consequence.
*   **Truncating (nonsense/splice site)**: Often lead to loss of function (LOF).

### Activity 1: Visual Inspection
Look at the plots of how mutations distribute along the gene, for your selected genes and consider the following:

1.  **Are mutations clustered in specific regions of the protein, or are they spread out?**

2.  **Does the gene have more missense or truncating mutations? What does this suggest about its role (oncogene vs. tumor suppressor)?**

3.  **Visually, compare the number of synonymous mutations (the "background") to the non-synonymous ones.**
    
    **Which ones are more frequent?**
    
    **How does this approximate ratio compare across the different genes you selected?**


In [ ]:
mutations_file = f"{input_data_path}/{SAMPLE_NAME}.somatic.mutations.tsv"
outdir = output_data_path

if os.path.exists(mutations_file):
    
    # Get counts
    counts_per_position = get_counts_per_position_n_consequence(mutations_file)

    for GENE_NAME in GENE_LIST:
        print(f"Processing gene: {GENE_NAME}")
        
        # Filter for selected gene
        gene_counts = counts_per_position[counts_per_position["Gene"] == GENE_NAME]
        print(gene_counts.groupby(by=["Consequence"])["Count"].sum().reset_index())

        if not gene_counts.empty:
            # 1. Needle Plot
            mut_count_df = gene_counts.groupby(by=["Pos", "Consequence"])["Count"].sum().reset_index()
            fig, ax = plt.subplots(1, 1, figsize=(6, 2))
            plot_count_track(
                mut_count_df,
                gene_len=mut_count_df["Pos"].max(), 
                axes=[ax], ax=0,
                colors_dict=metrics_colors_dictionary, indel=False, alpha=0.7
            )
            ax.set_title(f"{GENE_NAME} - Needle Plot")
            plt.show()

            # 2. Stacked Bar Plot
            fig, ax = plt.subplots(1, 1, figsize=(6, 2))
            plot_stacked_bar_track_binned(
                count_df=mut_count_df,
                gene_len=mut_count_df["Pos"].max(),
                axes=[ax], ax=0,
                colors_dict=metrics_colors_dictionary,
                alpha=1,
                indel=False
            )
            ax.set_title(f"{GENE_NAME} - Stacked Bar Plot")
            plt.show()
        else:
            print(f"No mutations found for gene {GENE_NAME}")
else:
    print(f"File {mutations_file} not found. Please ensure you have downloaded the data.")

## 3. Quantifying Selection Pressure

We will now apply three complementary methods to detect positive selection. Each looks at a different aspect of "selection":

#### A. Excess of mutations: $\omega$ (dN/dS) using [omega](https://github.com/bbglab/omega)
It calculates the ratio of observed non-synonymous mutations to expected ones (based on synonymous rates).
*   **$\omega = 1$**: Neutral evolution.
*   **$\omega > 1$**: Positive (diversifying) selection.
*   **$\omega < 1$**: Negative (purifying) selection.

#### B. Functional Impact Bias: [OncodriveFML](https://github.com/bbglab/oncodrivefml)
Detects genes where the observed mutations have a higher **functional impact score** (e.g., CADD) than expected by chance. This is dependent on the availability of scores and helps identify high-impact variants even if they aren't clustered.

#### C. Structural Clustering: [Oncodrive3D](https://github.com/bbglab/oncodrive3d)
Identifies genes where missense mutations cluster together in **3D space** on the protein surface, even if they are far apart in the 1D sequence. This metric will report strong signals for activating mutations in oncogenes and for functional regions 

### **Activity 2a: Metric Comparison**
Run the cells below to view the numerical tables and explanatory plots. Then, discuss in your group:

1. **Look at the explanatory plots for the dN/dS and functional impact bias selection metrics. Discuss the differences between genes.**



In [ ]:
omega_data = pd.read_table(f"{input_data_path}/all_omega_values.tsv")
omega_data[(omega_data["sample"] == 'all_samples')
           & (omega_data["gene"].isin(GENE_LIST))]

In [ ]:
oncodrivefml_data = pd.read_table(f"{input_data_path}/all_samples-oncodrivefml.tsv.gz")
oncodrivefml_data[oncodrivefml_data["SYMBOL"].isin(GENE_LIST)]

In [ ]:
# Generate "Side Figures" (Individual metric details)
# Note: This function generates plots for available metrics for the sample
print("Generating side figures...")
generate_all_side_figures(SAMPLE_NAME, outdir=outdir, gene_list=GENE_LIST, data_path = input_data_path)

### **Activity 2b: Metric Comparison**
Run the cell below to view the summary of the positive selection results per gene. Then, discuss in your group:

1.  **Do the metrics computed here agree with your expectations from the needle plots?**

2.  **Is the signal of selection consistent across the different metrics?** (is it consistently positively selected?)

3.  **Is the ranking of intensity of selection the same across the different metrics?** (look at the files in the Cancer_Genomic_Epidemiology_2026/course_data_2026/Clonal_selection_data/input/oncodrive3d_plots directory for more details on the regions with 3D clustering signal)




In [ ]:
# Generate Summary Plot (All tracks combined)
print("\nGenerating summary tracks...")
get_all_data(SAMPLE_NAME, outdir=outdir, tracks=("omega_trunc", "omega_mis", "oncodrive3d", "oncodrivefml"), gene_order=GENE_LIST, data_path = input_data_path)

*   **Top tracks**: Barplots for $\omega$ (dN/dS). Error bars indicate confidence intervals. If a bar is above 1 and the p-value is significant (filled color), the gene is positively selected.
*   **Bottom tracks**: Bubble plots for OncodriveFML and Oncodrive3D. The size of the bubble relates to the experimental score, and the color (filled) indicates significance.


## Summary so far

At this point we have all the information on which genes are important in the clonal structure of the bladder urothelium. However we are missing to understand which is the role of cancer risk factors in shaping this tissue.

For this we have to compare the samples with different exposures to cancer risk factors, and this is what we are going to do in the next section of the notebook.

## Activity 4: Compare neutral mutagenesis across samples

Before starting to compare the clonal selection metrics that may show differences between groups of samples, we can compare the background mutagenesis to detect any differences in the mutation rates.

### Mutation Density of non-protein affecting mutations

1. **Why does the non-protein affecting mutation density reflect background mutagenesis?**

2. **Does mutation density (non-protein affecting) show any correlation with age?**

3. **Are there significant (visually) differences in mutation rates between males and females for these genes?**

In [ ]:
# Load Mutation Density data with metadata
mut_density_file = f"{input_data_path}/all_mutation_densities.tsv"
metadata_file = f"{input_data_path}/metadata.tsv"

df_mut_density = pd.read_csv(mut_density_file, sep='\t')
df_mut_density = df_mut_density[df_mut_density['REGIONS'] == 'non_protein_affecting'].copy()
df_metadata = pd.read_csv(metadata_file, sep='\t')

# Merge datasets on SAMPLE_ID
df_merged = df_mut_density.merge(df_metadata, on='SAMPLE_ID')

# Cleaning: ensure mutation density is numeric and positive for log scale
df_merged['MUTDENSITY_MB'] = pd.to_numeric(df_merged['MUTDENSITY_MB'], errors='coerce')
df_merged = df_merged.dropna(subset=['MUTDENSITY_MB'])
df_merged = df_merged[df_merged['MUTDENSITY_MB'] > 0]


In [ ]:
print("--- Mutation Density Analysis (Non-Protein Affecting) ---")

for GENE_NAME in GENE_LIST:
    print(f"Analyzing {GENE_NAME}...")
    gene_subset = df_merged[df_merged['GENE'] == GENE_NAME].copy()
    
    if gene_subset.empty:
        print(f"No mutation density data found for {GENE_NAME}")
        continue

    # --- Age Correlation ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
    sns.scatterplot(data=gene_subset, x='AGE', y='MUTDENSITY_MB', alpha=0.7, ax=ax1)
    ax1.set_ylim(0, 14)
    ax1.set_title(f"Mutation Density (NPA) vs Age (Linear)")
    ax1.set_xlabel("Age")
    ax1.set_ylabel("Mutation Density (Mut/Mb)")
    ax1.grid(True, alpha=0.2)

    sns.scatterplot(data=gene_subset, x='AGE', y='MUTDENSITY_MB', alpha=0.7, ax=ax2)
    ax2.set_ylim(0.05, 14)
    ax2.set_yscale('log')
    ax2.set_title(f"Mutation Density (NPA) vs Age (Log)")
    ax2.set_xlabel("Age")
    ax2.set_ylabel("Mutation Density (log)")
    ax2.grid(True, which="both", alpha=0.2)
    
    ax1.legend().remove()
    ax2.legend().remove()
    
    fig.suptitle(GENE_NAME)
    plt.tight_layout()
    plt.show()

In [ ]:
print("--- Mutation Density Analysis (Non-Protein Affecting) ---")

for GENE_NAME in GENE_LIST:
    print(f"Analyzing {GENE_NAME}...")
    gene_subset = df_merged[df_merged['GENE'] == GENE_NAME].copy()
    
    if gene_subset.empty:
        print(f"No mutation density data found for {GENE_NAME}")
        continue

    # --- Sex Comparison (Sigmoid/Rank Plot) ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
    groups = []
    for sex in ['M', 'F']:
        sex_subset = gene_subset[gene_subset['SEX'] == sex].sort_values('MUTDENSITY_MB').reset_index(drop=True)
        if not sex_subset.empty:
            sex_subset['rank'] = sex_subset.index / len(sex_subset)
            groups.append(sex_subset)

    if groups:
        df_plot = pd.concat(groups)
        sns.scatterplot(data=df_plot, x='rank', y='MUTDENSITY_MB', hue='SEX', alpha=0.7, ax=ax1)
        ax1.set_ylim(0, 14)
        ax1.set_title(f"Mutation density (NPA) - Sex (Linear Scale)")
        ax1.set_xlabel("Relative Rank within Sex group")
        ax1.set_ylabel("Mutation Density (Mut/Mb)")
        ax1.grid(True, alpha=0.2)

        sns.scatterplot(data=df_plot, x='rank', y='MUTDENSITY_MB', hue='SEX', alpha=0.7, ax=ax2)
        ax2.set_yscale('log')
        ax2.set_title(f"Mutation density (NPA) - Sex (Log Scale)")
        ax2.set_xlabel("Relative Rank within Sex group")
        ax2.set_ylabel("Mutation Density (log)")
        ax2.grid(True, which="both", alpha=0.2)

        ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

    fig.suptitle(GENE_NAME)
    plt.tight_layout()
    plt.show()

## Activity 5: Compare metrics subject to selection across samples

### Activity 5.1: Compare protein-affecting mutation density across samples

1.  **Do you see any trend slope in protein-affecting mutation density values as age increases? How does it compare to that of non-protein-affecting?**

2.  **Do you see any trend slope in protein-affecting mutation density values when separating between males and females? How does it compare to the non-protein-affecting?**


In [ ]:
# Load Mutation Density data with metadata
mut_density_file = f"{input_data_path}/all_mutation_densities.tsv"
metadata_file = f"{input_data_path}/metadata.tsv"

df_mut_density = pd.read_csv(mut_density_file, sep='\t')
df_mut_density = df_mut_density[df_mut_density['REGIONS'] == 'protein_affecting'].copy()
df_metadata = pd.read_csv(metadata_file, sep='\t')

# Merge datasets on SAMPLE_ID
df_merged = df_mut_density.merge(df_metadata, on='SAMPLE_ID')

# Cleaning: ensure mutation density is numeric and positive for log scale
df_merged['MUTDENSITY_MB'] = pd.to_numeric(df_merged['MUTDENSITY_MB'], errors='coerce')
df_merged = df_merged.dropna(subset=['MUTDENSITY_MB'])
df_merged = df_merged[df_merged['MUTDENSITY_MB'] > 0]


In [ ]:
print("--- Mutation Density Analysis (Protein Affecting) ---")

for GENE_NAME in GENE_LIST:
    print(f"Analyzing {GENE_NAME}...")
    gene_subset = df_merged[df_merged['GENE'] == GENE_NAME].copy()
    
    if gene_subset.empty:
        print(f"No mutation density data found for {GENE_NAME}")
        continue

    # --- Age Correlation ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
    sns.scatterplot(data=gene_subset, x='AGE', y='MUTDENSITY_MB', alpha=0.7, ax=ax1)
    ax1.set_ylim(0, 14)
    ax1.set_title(f"Mutation Density (PA) vs Age (Linear)")
    ax1.set_xlabel("Age")
    ax1.set_ylabel("Mutation Density (Mut/Mb)")
    ax1.grid(True, alpha=0.2)

    sns.scatterplot(data=gene_subset, x='AGE', y='MUTDENSITY_MB', alpha=0.7, ax=ax2)
    ax2.set_ylim(0.05, 14)
    ax2.set_yscale('log')
    ax2.set_title(f"Mutation Density (PA) vs Age (Log)")
    ax2.set_xlabel("Age")
    ax2.set_ylabel("Mutation Density (log)")
    ax2.grid(True, which="both", alpha=0.2)
    
    ax1.legend().remove()
    ax2.legend().remove()
    
    fig.suptitle(GENE_NAME)
    plt.tight_layout()
    plt.show()

In [ ]:
print("--- Mutation Density Analysis (Non-Protein Affecting) ---")

for GENE_NAME in GENE_LIST:
    print(f"Analyzing {GENE_NAME}...")
    gene_subset = df_merged[df_merged['GENE'] == GENE_NAME].copy()
    
    if gene_subset.empty:
        print(f"No mutation density data found for {GENE_NAME}")
        continue

    # --- Sex Comparison (Sigmoid/Rank Plot) ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
    groups = []
    for sex in ['M', 'F']:
        sex_subset = gene_subset[gene_subset['SEX'] == sex].sort_values('MUTDENSITY_MB').reset_index(drop=True)
        if not sex_subset.empty:
            sex_subset['rank'] = sex_subset.index / len(sex_subset)
            groups.append(sex_subset)
    
    if groups:
        df_plot = pd.concat(groups)
        sns.scatterplot(data=df_plot, x='rank', y='MUTDENSITY_MB', hue='SEX', alpha=0.7, ax=ax1)
        ax1.set_title(f"Mutation density (PA) - Sex Comparison (Linear Scale)")
        ax1.set_xlabel("Relative Rank within Sex group")
        ax1.set_ylabel("Mutation Density (Mut/Mb)")
        ax1.grid(True, alpha=0.2)
        
        sns.scatterplot(data=df_plot, x='rank', y='MUTDENSITY_MB', hue='SEX', alpha=0.7, ax=ax2)
        ax2.set_yscale('log')
        ax2.set_title(f"Mutation density (PA) - Sex Comparison (Log Scale)")
        ax2.set_xlabel("Relative Rank within Sex group")
        ax2.set_ylabel("Mutation Density (log)")
        ax2.grid(True, which="both", alpha=0.2)
        
        ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.suptitle(GENE_NAME)
    plt.tight_layout()
    plt.show()

### Activity 5.2: Compare dN/dS values between males and females

1.  **Compare the distribution of values of dN/dS between the two groups. Do you see any difference in any of the selected genes?**

2.  **How do you interpret the observed differences? Is it only because there are more mutations in one group vs the other?**

3.  **Are there differences between the signal in missense and truncating dN/dS?**

In [ ]:
# Load Omega data with metadata
omega_file = f"{input_data_path}/all_omega_values.tsv"
metadata_file = f"{input_data_path}/metadata.tsv"

df_omega = pd.read_csv(omega_file, sep='\t')
df_metadata = pd.read_csv(metadata_file, sep='\t')

df_omega['gene_symbol'] = df_omega['gene']

# Merge datasets
df_merged = df_omega.merge(df_metadata, left_on='sample', right_on='SAMPLE_ID')

# Filter for 'missense' and 'truncating' impact
df_filtered = df_merged[df_merged['impact'].isin(['missense', 'truncating'])].copy()

# Cleaning: ensure dnds is numeric and positive for log scale
df_filtered['dnds'] = pd.to_numeric(df_filtered['dnds'], errors='coerce')
df_filtered = df_filtered.dropna(subset=['dnds'])
df_filtered = df_filtered[df_filtered['dnds'] > 0]

df_filtered_mis = df_filtered[df_filtered['impact'] == 'missense']
df_filtered_trunc = df_filtered[df_filtered['impact'] == 'truncating']

In [ ]:
print("--- Sex Comparison: Sigmoid Rank Distribution ---")
df_filtered = df_filtered_mis
for GENE_NAME in GENE_LIST:
    print(f"Analyzing {GENE_NAME}...")
    gene_subset = df_filtered[df_filtered['gene_symbol'] == GENE_NAME].copy()
    
    if gene_subset.empty:
        print(f"No data for {GENE_NAME}")
        continue

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))

    # Prep Sigmoid Data
    groups = []
    for sex in ['M', 'F']:
        sex_subset = gene_subset[gene_subset['SEX'] == sex].sort_values('dnds').reset_index(drop=True)
        if not sex_subset.empty:
            sex_subset['rank'] = sex_subset.index / len(sex_subset)
            groups.append(sex_subset)
    
    if groups:
        df_sex_plot = pd.concat(groups)
        
        # 1. Normal Scale
        sns.scatterplot(data=df_sex_plot, x='rank', y='dnds', hue='SEX', alpha=0.7, ax=ax1)
        ax1.axhline(y=1, color='gray', linestyle='--', alpha=0.6, label='Neutral (dN/dS=1)')
        ax1.set_title(f"Missense dN/dS - Sex Comparison (Linear Scale)")
        ax1.set_xlabel("Sorted omega values (Rank)")
        ax1.set_ylabel("dN/dS")
        ax1.grid(True, alpha=0.2)
        ax1.legend().remove()

        # 2. Log Scale
        sns.scatterplot(data=df_sex_plot, x='rank', y='dnds', hue='SEX', alpha=0.7, ax=ax2)
        ax2.set_yscale('log')
        ax2.axhline(y=1, color='gray', linestyle='--', alpha=0.6, label='Neutral (dN/dS=1)')
        ax2.set_title(f"Missense dN/dS - Sex Comparison (Log Scale)")
        ax2.set_xlabel("Sorted omega values (Rank)")
        ax2.set_ylabel("dN/dS (log)")
        ax2.grid(True, which="both", alpha=0.2)
        ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    else:
        ax1.set_title(f"No Sex data for {GENE_NAME}")

    fig.suptitle(GENE_NAME)
    plt.tight_layout()
    plt.show()


In [ ]:
print("--- Sex Comparison: Sigmoid Rank Distribution Truncating ---")
df_filtered = df_filtered_trunc
for GENE_NAME in GENE_LIST:
    print(f"Analyzing {GENE_NAME}...")
    gene_subset = df_filtered[df_filtered['gene_symbol'] == GENE_NAME].copy()
    
    if gene_subset.empty:
        print(f"No data for {GENE_NAME}")
        continue

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))

    # Prep Sigmoid Data
    groups = []
    for sex in ['M', 'F']:
        sex_subset = gene_subset[gene_subset['SEX'] == sex].sort_values('dnds').reset_index(drop=True)
        if not sex_subset.empty:
            sex_subset['rank'] = sex_subset.index / len(sex_subset)
            groups.append(sex_subset)
    
    if groups:
        df_sex_plot = pd.concat(groups)
        
        # 1. Normal Scale
        sns.scatterplot(data=df_sex_plot, x='rank', y='dnds', hue='SEX', alpha=0.7, ax=ax1)
        ax1.axhline(y=1, color='gray', linestyle='--', alpha=0.6, label='Neutral (dN/dS=1)')
        ax1.set_title(f"dN/dS truncating - Sex Comparison (Linear Scale)")
        ax1.set_xlabel("Sorted omega values (Rank)")
        ax1.set_ylabel("dN/dS")
        ax1.grid(True, alpha=0.2)
        ax1.legend().remove()

        # 2. Log Scale
        sns.scatterplot(data=df_sex_plot, x='rank', y='dnds', hue='SEX', alpha=0.7, ax=ax2)
        ax2.set_yscale('log')
        ax2.axhline(y=1, color='gray', linestyle='--', alpha=0.6, label='Neutral (dN/dS=1)')
        ax2.set_title(f"dN/dS truncating - Sex Comparison (Log Scale)")
        ax2.set_xlabel("Sorted omega values (Rank)")
        ax2.set_ylabel("dN/dS (log)")
        ax2.grid(True, which="both", alpha=0.2)
        ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    else:
        ax1.set_title(f"No Sex data for {GENE_NAME}")

    fig.suptitle(GENE_NAME)
    plt.tight_layout()
    plt.show()


## 6. Final Discussion & Wrap-up

Take a few minutes to synthesize your findings across the different sections.

### Final Questionnaire

1. **Which gene from your list is the one with the strongest positive selection?** 

2. **Is there anything that caught your attention when looking at the distribution of mutations along the gene and when interpreting the metrics?**

3. **How would you improve this qualitative analysis? i.e. how to make it quantitative? how representative it is? how could you investigate more risk factors?**

4. **Which new questions do you have now?**